# Use Case 1 — Load and Clean Chicago Crime Data

**Objective:** build a data-ingestion pipeline that **reads → cleans → enriches → stores** the
Chicago crime data in MySQL.

```
 CSV files ──► Pandas DataFrame ──► cleaning ──► new features ──► NumPy quality checks ──► MySQL
 (6 files)      (load + inspect)    (dates, NAs,   (Year, Month,     (% missing,             (normalized
                                     case, dupes)   DayOfWeek, Hour)  drop >50% cols)          schema)
```

Output of this notebook (used by notebooks 02–04):
* `output/chicago_crime_clean.pkl` – the cleaned DataFrame (pickle keeps datetime/bool types)
* MySQL database `chicago_crime_db` with 6 tables
* `output/data_quality_report.csv` – every anomaly we found and what we did about it

In [6]:
import numpy as np
import pandas as pd
from IPython.display import display

import crime_utils as cu

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
quality_log = []          # we append one row per data-quality finding


def log(check, rows_affected, action):
    quality_log.append({"check": check, "rows_affected": int(rows_affected), "action": action})
    print(f"[{check}] rows affected: {rows_affected:,}  ->  {action}")

## 1. Load the dataset into a Pandas DataFrame

`iucr_code`, `case_number` and `fbi_code` are **codes, not numbers** (e.g. `031A`, `08B`), so we
force them to `str`. If pandas guessed they were numbers it would silently turn `0486` into `486`
and crash on `031A`.

In [7]:
crime_path = cu.data_file("crime")

# A plain pd.read_csv(crime_path) FAILS on this file:
#   "ParserError: Expected 20 fields in line 498, saw 21"
# The last lines end with an unclosed quote  "(41.76..., -87.64...)   <- no closing "
# (the export was truncated), so the parser merges them into one broken row.
# cu.read_csv_repaired closes such quotes before parsing.
raw, repaired_lines = cu.read_csv_repaired(
    crime_path, dtype={"iucr_code": str, "case_number": str, "fbi_code": str}, low_memory=False)
log("Line with unclosed quote (truncated export)", len(repaired_lines),
    f"quote closed on file lines {repaired_lines}")

# normalise column names once: strip spaces, lower-case, snake_case
raw.columns = raw.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)

[Line with unclosed quote (truncated export)] rows affected: 2  ->  quote closed on file lines [498, 499]


### 1a. First 10 rows

In [8]:
display(raw.head(10))

,id,case_number,date,block,iucr_code,location_desc,arrest,domestic,beat_num,district_code,ward_no,community_code,fbi_code,x_coordinate,y_coordinate,year,date_of_update,latitude,longitude,location
0,10508693,HZ250496,5/3/2016,013XX S SAWYER AVE,486,APARTMENT,True,True,1022,10,24,29,08B,1154907.0,1893681.0,2016,5/10/2016,41.864073,-87.706819,"(41.864073157, -87.706818608)"
1,10508695,HZ250409,5/3/2016,061XX S DREXEL AVE,486,RESIDENCE,False,True,313,3,20,42,08B,1183066.0,1864330.0,2016,5/10/2016,41.782922,-87.604363,"(41.782921527, -87.60436317)"
2,10508697,HZ250503,5/3/2016,053XX W CHICAGO AVE,470,STREET,False,False,1524,15,37,25,24,1140789.0,1904819.0,2016,5/10/2016,41.894908,-87.758372,"(41.894908283, -87.758371958)"
3,10508698,HZ250424,5/3/2016,049XX W FULTON ST,460,SIDEWALK,False,False,1532,15,28,25,08B,1143223.0,1901475.0,2016,5/10/2016,41.885687,-87.749516,"(41.885686845, -87.749515983)"
4,10508699,HZ250455,5/3/2016,003XX N LOTUS AVE,820,RESIDENCE,False,True,1523,15,28,25,6,1139890.0,1901675.0,2016,5/10/2016,41.886297,-87.761751,"(41.886297242, -87.761750709)"
5,10508702,HZ250447,5/3/2016,082XX S MARYLAND AVE,041A,STREET,False,False,631,6,8,44,04B,1183336.0,1850642.0,2016,5/10/2016,41.745354,-87.603799,"(41.745354023, -87.603798903)"
6,10508703,HZ250489,5/3/2016,027XX S STATE ST,460,CHA HALLWAY/STAIRWELL/ELEVATOR,False,False,133,1,3,35,08B,1176730.0,1886544.0,2016,5/10/2016,41.844024,-87.626923,"(41.844023772, -87.626923253)"
7,10508704,HZ250514,5/3/2016,002XX E 46TH ST,460,RESIDENCE PORCH/HALLWAY,False,False,215,2,3,38,08B,1178514.0,1874573.0,2016,5/10/2016,41.811134,-87.620741,"(41.811133958, -87.62074077)"
8,10508709,HZ250523,5/3/2016,014XX W DEVON AVE,460,SIDEWALK,False,False,2432,24,40,1,08B,1165696.0,1942616.0,2016,5/10/2016,41.998131,-87.665814,"(41.99813061, -87.665814038)"
9,10508982,HZ250667,5/3/2016,069XX S ASHLAND AVE,486,STREET,False,True,735,7,17,67,08B,1166876.0,1858796.0,2016,5/10/2016,41.768097,-87.663879,"(41.768096835, -87.663878589)"


### 1b. Schema and data types
`df.info()` shows each column, its non-null count and its dtype. Notice:
* `date` and `date_of_update` are **object** (text) — they must be converted to datetime.
* `arrest` / `domestic` are text (`TRUE`/`FALSE`), not booleans.
* `ward_no`, `community_code`, `district_code` became **float64** — pandas does this whenever an
  integer column contains NaN. We'll fix that with the nullable `Int64` type.

In [9]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 498 entries, 0 to 497
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              498 non-null    int64  
 1   case_number     498 non-null    str    
 2   date            498 non-null    str    
 3   block           498 non-null    str    
 4   iucr_code       498 non-null    str    
 5   location_desc   494 non-null    str    
 6   arrest          498 non-null    bool   
 7   domestic        498 non-null    bool   
 8   beat_num        498 non-null    int64  
 9   district_code   498 non-null    int64  
 10  ward_no         498 non-null    int64  
 11  community_code  498 non-null    int64  
 12  fbi_code        498 non-null    str    
 13  x_coordinate    469 non-null    float64
 14  y_coordinate    469 non-null    float64
 15  year            498 non-null    int64  
 16  date_of_update  498 non-null    str    
 17  latitude        469 non-null    float64
 18  l

In [10]:
schema = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "non_null": raw.notna().sum(),
    "nulls": raw.isna().sum(),
    "unique_values": raw.nunique(),
    "example": raw.iloc[0],
})
display(schema)

,dtype,non_null,nulls,unique_values,example
id,int64,498,0,498,10508693
case_number,str,498,0,498,HZ250496
date,str,498,0,46,5/3/2016
block,str,498,0,475,013XX S SAWYER AVE
iucr_code,str,498,0,97,486
location_desc,str,494,4,47,APARTMENT
arrest,bool,498,0,2,True
domestic,bool,498,0,2,True
beat_num,int64,498,0,209,1022
district_code,int64,498,0,22,10


### 1c. How many rows and columns?

In [11]:
n_rows, n_cols = raw.shape
print(f"Rows: {n_rows:,}   Columns: {n_cols}")

Rows: 498   Columns: 20


### 1d. Load the 5 lookup (dimension) files
These are the tables referenced by foreign keys in the data model.

The delivered files don't match the data model exactly, so we fix the headers on load:
* `iucr_codes.csv.csv` → header `IUCR, PRIMARY_TYPE␣, DESCRIPTION, INDEX CODE` (note the trailing
  space and the space in `INDEX CODE`) → normalised to `iucr_code, primary_type, description, index_code`
* `chicago_ward_offices.csv` → the key column is called `ward`, renamed to `ward_no`

In [12]:
RENAME = {"iucr": "iucr_code", "ward": "ward_no"}

def load_lookup(key):
    d = pd.read_csv(cu.data_file(key), dtype=str)
    d.columns = (d.columns.str.strip().str.lower()
                   .str.replace(r"\s+", "_", regex=True))   # 'PRIMARY_TYPE ' -> 'primary_type'
    d = d.rename(columns=RENAME)
    return d.apply(lambda s: s.str.strip())          # strip whitespace in every cell

iucr_raw = load_lookup("iucr")
community_raw = load_lookup("community")
district_raw = load_lookup("district")
ward_raw = load_lookup("ward")
beat_raw = load_lookup("beat")

for name, d in [("iucr", iucr_raw), ("community", community_raw), ("district", district_raw),
                ("ward", ward_raw), ("beat", beat_raw)]:
    print(f"{name:<10} {d.shape[0]:>4} rows x {d.shape[1]:>2} cols   {list(d.columns)[:6]} ...")

iucr        410 rows x  4 cols   ['iucr_code', 'primary_type', 'description', 'index_code'] ...
community    77 rows x  7 cols   ['community_code', 'community_name', 'population', 'area_sqmile', 'area_sqkm', 'density_per_sqmi'] ...
district     22 rows x 15 cols   ['district_code', 'district_name', 'address', 'city', 'state', 'zip'] ...
ward         58 rows x 20 cols   ['ward_no', 'alderman', 'address', 'city', 'state', 'zipcode'] ...
beat        277 rows x  4 cols   ['district', 'sector', 'beat', 'beat_num'] ...


## 2. Clean the dataset

We work on a copy (`df`) so `raw` stays untouched for before/after comparisons.

### 2a. Remove duplicates
* **Exact duplicates** — the same row twice (typically a double export).
* **Duplicate `case_number`** — it's the primary key, so it must be unique. When a case appears
  twice we keep the version with the most recent `date_of_update`.

In [13]:
df = raw.copy()

n_exact = df.duplicated().sum()
df = df.drop_duplicates()
log("Exact duplicate rows", n_exact, "dropped")

df["_upd"] = pd.to_datetime(df["date_of_update"], format="%m/%d/%Y %H:%M", errors="coerce")
n_case_dup = df.duplicated("case_number").sum()
df = (df.sort_values("_upd").drop_duplicates("case_number", keep="last")
        .drop(columns="_upd").sort_index())
log("Duplicate case_number (PK)", n_case_dup, "kept the most recently updated record")

n_no_case = df["case_number"].isna().sum()
df = df[df["case_number"].notna()]
log("Missing case_number", n_no_case, "dropped (cannot be stored without a primary key)")

[Exact duplicate rows] rows affected: 0  ->  dropped
[Duplicate case_number (PK)] rows affected: 0  ->  kept the most recently updated record
[Missing case_number] rows affected: 0  ->  dropped (cannot be stored without a primary key)


### 2b. Convert the `date` column into datetime format

The expected format is `5/3/2016 23:40` → `%m/%d/%Y %H:%M`. Real exports often mix formats, so
instead of one `pd.to_datetime` call (which would turn every non-matching value into `NaT`
silently) we **try a list of known formats in order** and record which one matched each row.
Anything that matches none of them is a genuine anomaly.

In [14]:
DATE_FORMATS = [
    "%m/%d/%Y %H:%M",          # 5/3/2016 23:40         (expected)
    "%m/%d/%Y %I:%M:%S %p",    # 05/03/2016 11:40:00 PM (Chicago data-portal export)
    "%m/%d/%Y %H:%M:%S",       # 05/03/2016 23:40:00
    "%Y-%m-%d %H:%M:%S",       # 2016-05-03 23:40:00    (ISO)
    "%Y-%m-%d %H:%M",
    "%m/%d/%Y",                # date only
]


def parse_mixed_dates(series):
    """Parse a text column that may contain several date formats.
    Returns (parsed_datetimes, format_used_per_row)."""
    s = series.astype("string").str.strip()
    parsed = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    fmt_used = pd.Series(pd.NA, index=s.index, dtype="string")
    for fmt in DATE_FORMATS:
        todo = parsed.isna() & s.notna()
        if not todo.any():
            break
        attempt = pd.to_datetime(s[todo], format=fmt, errors="coerce")
        ok = attempt.dropna().index
        parsed.loc[ok] = attempt.loc[ok]
        fmt_used.loc[ok] = fmt
    return parsed, fmt_used


df["date_parsed"], df["date_format"] = parse_mixed_dates(df["date"])
print("Which date formats were found:")
display(df["date_format"].value_counts(dropna=False).rename("rows").to_frame())

Which date formats were found:


,rows
date_format,
%m/%d/%Y,498


In [15]:
bad_dates = df[df["date_parsed"].isna()]
print("Examples of values that could not be parsed:")
display(bad_dates["date"].value_counts(dropna=False).head(10).rename("rows").to_frame())

dominant = df["date_format"].mode().iloc[0]
n_alt = (df["date_format"].notna() & (df["date_format"] != dominant)).sum()
print(f"Dominant format: {dominant}")
log("Date in a different format from the rest", n_alt, "parsed with the matching format")

date_only = df["date_format"].isin(["%m/%d/%Y"])
log("Date has no time of day (date only)", date_only.sum(),
    "time set to 00:00 – hour-of-day analysis is NOT possible for these rows")
log("Date missing / unparseable", len(bad_dates), "dropped (no time information = unusable)")
df = df[df["date_parsed"].notna()].copy()

Examples of values that could not be parsed:


,rows
date,


Dominant format: %m/%d/%Y
[Date in a different format from the rest] rows affected: 0  ->  parsed with the matching format
[Date has no time of day (date only)] rows affected: 498  ->  time set to 00:00 – hour-of-day analysis is NOT possible for these rows
[Date missing / unparseable] rows affected: 0  ->  dropped (no time information = unusable)


**Logical date checks** — a value can parse correctly and still be wrong:
* a crime date **in the future**;
* a date **after** the record's `date_of_update` (a record can't be updated before it happened);
* the `year` column **disagreeing** with the year inside `date`.

In [16]:
df["date"] = df.pop("date_parsed")
df["date_of_update"], _ = parse_mixed_dates(df["date_of_update"])
df = df.drop(columns="date_format")

today = pd.Timestamp.today().normalize()
future = df["date"] > today
log("Crime date in the future", future.sum(), "dropped")
df = df[~future]

upd_before = df["date_of_update"].notna() & (df["date_of_update"] < df["date"])
log("date_of_update earlier than date", upd_before.sum(), "flagged only (kept)")

year_col = pd.to_numeric(df["year"], errors="coerce")
year_mismatch = year_col != df["date"].dt.year
log("year column disagrees with date", year_mismatch.sum(), "year rebuilt from date")
df["year"] = df["date"].dt.year

print(f"\nDate range after cleaning: {df['date'].min()}  →  {df['date'].max()}")

[Crime date in the future] rows affected: 0  ->  dropped
[date_of_update earlier than date] rows affected: 0  ->  flagged only (kept)
[year column disagrees with date] rows affected: 0  ->  year rebuilt from date

Date range after cleaning: 2012-01-01 00:00:00  →  2016-05-04 00:00:00


### 2c. Standardize categorical fields (strip whitespace, unify case)

`"APARTMENT"`, `"Apartment"` and `"  apartment "` are the same category, but pandas would count
them as three. We strip, collapse repeated spaces and upper-case every text column.

Codes get extra treatment: IUCR codes are 4 characters in the official list (`0486`), but many
files drop the leading zero (`486`). We pad both the crime file **and** the lookup with
`zfill(4)` so they join correctly.

In [17]:
def standardize_text(s):
    return (s.astype("string").str.strip()
             .str.replace(r"\s+", " ", regex=True).str.upper())

before = df["location_desc"].nunique()
for col in ["block", "location_desc", "fbi_code", "iucr_code", "case_number"]:
    df[col] = standardize_text(df[col])
print(f"location_desc distinct values: {before} before → {df['location_desc'].nunique()} after")

df["iucr_code"] = df["iucr_code"].str.zfill(4)
iucr = iucr_raw.copy()
iucr["iucr_code"] = standardize_text(iucr["iucr_code"]).str.zfill(4)
for col in ["primary_type", "description", "index_code"]:
    iucr[col] = standardize_text(iucr[col])
iucr = iucr.drop_duplicates("iucr_code")

location_desc distinct values: 47 before → 47 after


**Booleans** — `arrest` and `domestic` arrive as text (`TRUE`, `true`, `FALSE`…). We map every
common spelling to a real `bool`.

In [18]:
BOOL_MAP = {"TRUE": True, "T": True, "Y": True, "YES": True, "1": True, "1.0": True,
            "FALSE": False, "F": False, "N": False, "NO": False, "0": False, "0.0": False}

for col in ["arrest", "domestic"]:
    txt = standardize_text(df[col])
    mapped = txt.map(BOOL_MAP)
    unmapped = txt.notna() & mapped.isna()
    if unmapped.any():
        print(f"{col}: unexpected values {txt[unmapped].unique()[:5]}")
    log(f"{col} missing/unrecognised", mapped.isna().sum(),
        "set to False (no record of an arrest/domestic flag)")
    df[col] = mapped.fillna(False).astype(bool)

print(df[["arrest", "domestic"]].dtypes)

[arrest missing/unrecognised] rows affected: 0  ->  set to False (no record of an arrest/domestic flag)
[domestic missing/unrecognised] rows affected: 0  ->  set to False (no record of an arrest/domestic flag)
arrest      bool
domestic    bool
dtype: object


**Numeric codes** — `beat_num`, `district_code`, `ward_no`, `community_code` and the coordinates
are converted with `pd.to_numeric(errors="coerce")`. Codes use the **nullable `Int64`** type, so a
missing ward stays `<NA>` instead of turning the whole column into floats.

In [19]:
for col in ["beat_num", "district_code", "ward_no", "community_code", "x_coordinate", "y_coordinate", "id"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").round().astype("Int64")
for col in ["latitude", "longitude"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

**Join the IUCR lookup** to get `primary_type` (the crime category) and `description`.
The master file only stores the code; the category lives in `iucr_codes.csv`.

In [20]:
if "primary_type" in df.columns:          # some versions of the file already include it
    df = df.drop(columns=[c for c in ["primary_type", "description"] if c in df.columns])
df = df.merge(iucr[["iucr_code", "primary_type", "description"]], on="iucr_code", how="left")

unknown_iucr = df["primary_type"].isna()
print("IUCR codes not found in lookup:", df.loc[unknown_iucr, "iucr_code"].unique())
log("IUCR code not in lookup table", unknown_iucr.sum(), "primary_type = 'UNKNOWN'")
df["primary_type"] = df["primary_type"].fillna("UNKNOWN")
df["description"] = df["description"].fillna("UNKNOWN")

IUCR codes not found in lookup: <StringArray>
[]
Length: 0, dtype: string
[IUCR code not in lookup table] rows affected: 0  ->  primary_type = 'UNKNOWN'


### 2d. Identify and handle missing values for key fields

There is no single right answer for missing data — each column needs a decision based on what
it means. The rule used here: **never invent information we can't justify.**

| Field | Strategy | Why |
|---|---|---|
| `case_number`, `date` | drop row | already done — primary key / no time = unusable |
| `iucr_code` | drop row | no code = no crime category |
| `district_code` | **derive from `beat_num`** via the beat lookup | a beat belongs to exactly one district, so this is a fact, not a guess |
| `location_desc` | fill `"UNKNOWN"` | keeps the row; makes "unknown" explicit in charts |
| `ward_no`, `community_code` | keep as `NULL` | guessing an area would corrupt area-level analysis; could be recovered later by a spatial join on lat/long |
| `latitude`/`longitude`/`x`/`y` | keep as `NULL` | not needed for counts; never fabricate locations |

In [21]:
key_fields = ["case_number", "date", "iucr_code", "primary_type", "arrest", "domestic",
              "location_desc", "beat_num", "district_code", "ward_no", "community_code",
              "latitude", "longitude"]
missing_before = df[key_fields].isna().sum().rename("missing_before")

# iucr_code
n = df["iucr_code"].isna().sum()
df = df[df["iucr_code"].notna()]
log("Missing iucr_code", n, "dropped")

# district from beat (lookup first, fall back to beat // 100 which is how Chicago numbers beats)
beat_lu = beat_raw.assign(beat_num=pd.to_numeric(beat_raw["beat_num"], errors="coerce"),
                          district=pd.to_numeric(beat_raw["district"], errors="coerce"))
beat_to_district = beat_lu.drop_duplicates("beat_num").set_index("beat_num")["district"]
miss_d = df["district_code"].isna() & df["beat_num"].notna()
derived = df.loc[miss_d, "beat_num"].map(beat_to_district)
derived = derived.fillna(df.loc[miss_d, "beat_num"] // 100)
df.loc[miss_d, "district_code"] = derived.astype("Int64")
log("Missing district_code", miss_d.sum(), "derived from beat_num")

n = df["location_desc"].isna().sum()
df["location_desc"] = df["location_desc"].fillna("UNKNOWN")
log("Missing location_desc", n, "filled with 'UNKNOWN'")

for col in ["ward_no", "community_code"]:
    log(f"Missing {col}", df[col].isna().sum(), "kept as NULL")

missing_after = df[key_fields].isna().sum().rename("missing_after")
display(pd.concat([missing_before, missing_after], axis=1))

[Missing iucr_code] rows affected: 0  ->  dropped
[Missing district_code] rows affected: 0  ->  derived from beat_num
[Missing location_desc] rows affected: 4  ->  filled with 'UNKNOWN'
[Missing ward_no] rows affected: 0  ->  kept as NULL
[Missing community_code] rows affected: 0  ->  kept as NULL


,missing_before,missing_after
case_number,0,0
date,0,0
iucr_code,0,0
primary_type,0,0
arrest,0,0
domestic,0,0
location_desc,4,0
beat_num,0,0
district_code,0,0
ward_no,0,0


### 2e. Coordinate outliers
Chicago lies roughly within latitude **41.64 – 42.03** and longitude **−87.95 – −87.52**.
* A **positive longitude** with the right magnitude (e.g. `87.70`) is a missing minus sign → we
  flip the sign.
* Anything else outside the box (e.g. `36.6, −91.7`, which is in Missouri) is an error → `NULL`.

In [22]:
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 41.64, 42.03, -87.95, -87.52

sign_flip = df["longitude"].between(-LON_MAX, -LON_MIN)
df.loc[sign_flip, "longitude"] = -df.loc[sign_flip, "longitude"]
log("Longitude with missing minus sign", sign_flip.sum(), "sign corrected")

has_coords = df["latitude"].notna() & df["longitude"].notna()
outside = has_coords & ~(df["latitude"].between(LAT_MIN, LAT_MAX) & df["longitude"].between(LON_MIN, LON_MAX))
print(df.loc[outside, ["case_number", "latitude", "longitude"]].head())
log("Coordinates outside Chicago", outside.sum(), "lat/long/x/y set to NULL")
df.loc[outside, ["latitude", "longitude", "x_coordinate", "y_coordinate"]] = np.nan

# rebuild the 'location' text from the corrected coordinates
df["location"] = np.where(df["latitude"].notna(),
                          "(" + df["latitude"].round(8).astype(str) + ", " + df["longitude"].round(8).astype(str) + ")",
                          None)

[Longitude with missing minus sign] rows affected: 0  ->  sign corrected
Empty DataFrame
Columns: [case_number, latitude, longitude]
Index: []
[Coordinates outside Chicago] rows affected: 0  ->  lat/long/x/y set to NULL


## 3. Generate new features
The `.dt` accessor works only on datetime columns — which is why step 2b had to come first.
`Hour` is also created here because Use Case 3 needs it.

In [23]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.day_name()
df["hour"] = df["date"].dt.hour

display(df[["date", "year", "month", "day_of_week", "hour"]].head())

,date,year,month,day_of_week,hour
0,2016-05-03,2016,5,Tuesday,0
1,2016-05-03,2016,5,Tuesday,0
2,2016-05-03,2016,5,Tuesday,0
3,2016-05-03,2016,5,Tuesday,0
4,2016-05-03,2016,5,Tuesday,0


## 4. Use NumPy — percentage of missing values per column

`df.isna().to_numpy()` gives a 2-D boolean array (`True` = missing). Booleans count as 1/0, so
`np.mean(..., axis=0)` (mean down each column) is exactly the **fraction missing** per column.
We compare the raw file with the cleaned data.

In [24]:
def missing_pct_numpy(frame):
    mask = frame.isna().to_numpy()                    # shape (rows, cols), dtype=bool
    pct = np.mean(mask, axis=0) * 100                 # column-wise mean -> fraction missing
    return pd.Series(np.round(pct, 3), index=frame.columns)

missing_report = pd.DataFrame({
    "raw_missing_%": missing_pct_numpy(raw),
    "clean_missing_%": missing_pct_numpy(df.reindex(columns=raw.columns)),
}).sort_values("raw_missing_%", ascending=False)
display(missing_report)

,raw_missing_%,clean_missing_%
location,5.823,5.823
y_coordinate,5.823,5.823
longitude,5.823,5.823
latitude,5.823,5.823
x_coordinate,5.823,5.823
location_desc,0.803,0.000
case_number,0.000,0.000
id,0.000,0.000
block,0.000,0.000
date,0.000,0.000


### Can we drop columns with > 50 % missing?
Yes — `df.columns[pct > 50]` gives the list, and `df.drop(columns=...)` removes them. But we
**protect key fields**: if `date` or `primary_type` were 60 % empty, dropping the column would
destroy the analysis — that would need investigating, not deleting.

In [25]:
THRESHOLD = 50
PROTECTED = {"id", "case_number", "date", "iucr_code", "primary_type", "arrest", "domestic"}

pct_now = missing_pct_numpy(df)
over = pct_now[pct_now > THRESHOLD].index.tolist()
to_drop = [c for c in over if c not in PROTECTED]
blocked = [c for c in over if c in PROTECTED]

print(f"Columns with > {THRESHOLD}% missing: {over or 'none'}")
if blocked:
    print(f"NOT dropped (protected key fields, investigate instead): {blocked}")
df = df.drop(columns=to_drop)
print(f"Dropped: {to_drop or 'nothing'} | Highest missing % in any column: "
      f"{pct_now.max():.2f}% ({pct_now.idxmax()})")

Columns with > 50% missing: none
Dropped: nothing | Highest missing % in any column: 5.82% (x_coordinate)


## 5. Insert cleaned data into MySQL

### 5a. Schema design
The schema (`sql/01_create_schema.sql`) follows the data model — a **star schema**:

* `chicago_crime` is the **fact table** (one row per incident).
* `iucr`, `city_community`, `district_ps_info`, `ward_office`, `police_beat_info` are
  **dimension/lookup tables**, linked by **foreign keys**.
* `primary_type` lives only in `iucr` (normalisation: store each fact once; join when needed).
* `year`, `month`, `day_of_week`, `hour` are stored on the fact table — a deliberate
  denormalisation so reporting queries don't recompute them.
* Indexes on `year`, `(year, month)` and `community_code` speed up the Use Case 4 queries.

### 5b. Write with SQLAlchemy + PyMySQL
Load order matters: **lookup tables first**, then the fact table — otherwise MySQL rejects rows
whose foreign key points at something that doesn't exist yet.

In [26]:
from sqlalchemy import text

server = cu.get_engine(with_db=False)
with server.begin() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {cu.DB['database']} "
                      "CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci"))

engine = cu.get_engine()
n_stmt = cu.run_sql_file(engine, cu.SQL_DIR / "01_create_schema.sql")
print(f"Schema created ({n_stmt} statements)")

Schema created (14 statements)


**Prepare the lookup tables** — convert text like `"55,062"` into numbers, and make sure every
code used by the crime table exists in its lookup.

In [27]:
def to_num(s, kind=float):
    v = pd.to_numeric(s.astype("string").str.replace(",", "", regex=False), errors="coerce")
    return v.round().astype("Int64") if kind is int else v

community = community_raw.copy()
community["community_code"] = to_num(community["community_code"], int)
community["population"] = to_num(community["population"], int)
for c in ["area_sqmile", "area_sqkm", "density_per_sqmi", "density_per_sqkm"]:
    community[c] = to_num(community[c])
community["community_name"] = standardize_text(community["community_name"])
community = community.dropna(subset=["community_code"]).drop_duplicates("community_code")

district = district_raw.copy()
district["district_code"] = to_num(district["district_code"], int)
for c in ["x_coordinate", "y_coordinate", "latitude", "longitude"]:
    district[c] = to_num(district[c])
district = district.dropna(subset=["district_code"]).drop_duplicates("district_code")

ward = ward_raw.copy()
ward["ward_no"] = to_num(ward["ward_no"], int)
ward = ward.dropna(subset=["ward_no"])
# Some wards appear twice: an older office record and a newer one appended at the END of the
# file (e.g. ward 30 Reboyras -> Cruz, ward 6 Sawyer -> Hall). We keep the LAST occurrence.
dup_wards = sorted(ward.loc[ward["ward_no"].duplicated(keep=False), "ward_no"].unique().tolist())
log("Duplicate ward_no in ward offices file", ward["ward_no"].duplicated().sum(),
    f"kept the last (newest) record for wards {dup_wards}")
ward = ward.drop_duplicates("ward_no", keep="last")

beat = beat_raw.copy()
for c in ["beat_num", "district", "sector", "beat"]:
    beat[c] = to_num(beat[c], int)
beat = beat.dropna(subset=["beat_num"])
log("Duplicate beat_num in police beat file", beat["beat_num"].duplicated().sum(),
    "identical rows – duplicates dropped")
beat = beat.drop_duplicates("beat_num")

# every IUCR code in the crime data must exist in iucr (FK) -> add placeholders for unknown ones
missing_codes = sorted(set(df["iucr_code"]) - set(iucr["iucr_code"]))
iucr_db = pd.concat([iucr, pd.DataFrame({"iucr_code": missing_codes, "primary_type": "UNKNOWN",
                                         "description": "CODE NOT IN LOOKUP", "index_code": None})],
                    ignore_index=True)
print("Placeholder IUCR codes added:", missing_codes)

[Duplicate ward_no in ward offices file] rows affected: 8  ->  kept the last (newest) record for wards [2, 6, 20, 27, 30, 31, 35, 45]
[Duplicate beat_num in police beat file] rows affected: 2  ->  identical rows – duplicates dropped
Placeholder IUCR codes added: []


**Orphan foreign keys** — if a crime references a beat/ward/district/community that isn't in the
lookup file, MySQL would reject the row. We set those references to `NULL` and log how many.

In [28]:
fk_checks = {"beat_num": beat["beat_num"], "district_code": district["district_code"],
             "ward_no": ward["ward_no"], "community_code": community["community_code"]}
for col, valid in fk_checks.items():
    orphan = df[col].notna() & ~df[col].isin(valid.dropna())
    if orphan.any():
        df.loc[orphan, col] = pd.NA
    log(f"Orphan FK {col}", orphan.sum(), "set to NULL")

[Orphan FK beat_num] rows affected: 0  ->  set to NULL
[Orphan FK district_code] rows affected: 0  ->  set to NULL
[Orphan FK ward_no] rows affected: 0  ->  set to NULL
[Orphan FK community_code] rows affected: 0  ->  set to NULL


In [33]:
CRIME_COLS = ["id", "case_number", "date", "block", "iucr_code", "location_desc", "arrest",
              "domestic", "beat_num", "district_code", "ward_no", "community_code", "fbi_code",
              "x_coordinate", "y_coordinate", "year", "month", "day_of_week", "hour",
              "date_of_update", "latitude", "longitude", "location"]

crime_db = df[[c for c in CRIME_COLS if c in df.columns]].copy()
crime_db["arrest"] = crime_db["arrest"].astype(int)
crime_db["domestic"] = crime_db["domestic"].astype(int)
crime_db = crime_db.astype(object).where(crime_db.notna(), None)   # NaN/NA -> SQL NULL

loads = [("iucr", iucr_db), ("city_community", community), ("district_ps_info", district),
         ("ward_office", ward), ("police_beat_info", beat), ("chicago_crime", crime_db)]

for table, frame in loads:
    cols = pd.read_sql(text(f"SELECT * FROM {table} LIMIT 0"), engine).columns
    frame = frame[[c for c in frame.columns if c in cols]]
    frame.to_sql(table, engine, if_exists="append", index=False, chunksize=5000, method="multi")
    print(f"Loaded {len(frame):>7,} rows into {table}")

DatabaseError: Execution failed on sql 'INSERT INTO iucr (iucr_code, primary_type, description, index_code) VALUES (:iucr_code_m0, :primary_type_m0, :description_m0, :index_code_m0), (:iucr_code_m1, :primary_type_m1, :description_m1, :index_code_m1), (:iucr_code_m2, :primary_type_m2, :description_m2, :index_code_m2), (:iucr_code_m3, :primary_type_m3, :description_m3, :index_code_m3), (:iucr_code_m4, :primary_type_m4, :description_m4, :index_code_m4), (:iucr_code_m5, :primary_type_m5, :description_m5, :index_code_m5), (:iucr_code_m6, :primary_type_m6, :description_m6, :index_code_m6), (:iucr_code_m7, :primary_type_m7, :description_m7, :index_code_m7), (:iucr_code_m8, :primary_type_m8, :description_m8, :index_code_m8), (:iucr_code_m9, :primary_type_m9, :description_m9, :index_code_m9), (:iucr_code_m10, :primary_type_m10, :description_m10, :index_code_m10), (:iucr_code_m11, :primary_type_m11, :description_m11, :index_code_m11), (:iucr_code_m12, :primary_type_m12, :description_m12, :index_

**Verify** — row counts in MySQL must match the DataFrames.

In [25]:
counts = pd.read_sql(text("""
    SELECT 'chicago_crime' AS table_name, COUNT(*) AS rows_in_mysql FROM chicago_crime
    UNION ALL SELECT 'iucr', COUNT(*) FROM iucr
    UNION ALL SELECT 'city_community', COUNT(*) FROM city_community
    UNION ALL SELECT 'district_ps_info', COUNT(*) FROM district_ps_info
    UNION ALL SELECT 'ward_office', COUNT(*) FROM ward_office
    UNION ALL SELECT 'police_beat_info', COUNT(*) FROM police_beat_info"""), engine)
display(counts)
assert counts.loc[counts.table_name == "chicago_crime", "rows_in_mysql"].item() == len(df)
print("✔ Row counts match")

,table_name,rows_in_mysql
0,chicago_crime,498
1,iucr,410
2,city_community,77
3,district_ps_info,22
4,ward_office,50
5,police_beat_info,275


✔ Row counts match


## Questions

### Q1. How many unique crime types exist in the dataset?
"Crime type" = `primary_type` (the IUCR category). We also show the finer-grained IUCR codes.

In [26]:
known = df.loc[df["primary_type"] != "UNKNOWN", "primary_type"]
print(f"Unique crime types (primary_type): {known.nunique()}")
print(f"Unique IUCR codes (sub-types):     {df['iucr_code'].nunique()}")
print(f"Rows with an unrecognised code:    {(df['primary_type'] == 'UNKNOWN').sum()}")
print("\nCrime types:", ", ".join(sorted(known.unique())))

Unique crime types (primary_type): 23
Unique IUCR codes (sub-types):     97
Rows with an unrecognised code:    0

Crime types: ASSAULT, BATTERY, BURGLARY, CRIMINAL DAMAGE, CRIMINAL SEXUAL ASSAULT, CRIMINAL TRESPASS, DECEPTIVE PRACTICE, GAMBLING, HOMICIDE, INTERFERENCE WITH PUBLIC OFFICER, KIDNAPPING, MOTOR VEHICLE THEFT, NARCOTICS, OBSCENITY, OFFENSE INVOLVING CHILDREN, OTHER OFFENSE, PROSTITUTION, PUBLIC PEACE VIOLATION, ROBBERY, SEX OFFENSE, STALKING, THEFT, WEAPONS VIOLATION


### Q2. Are there anomalies in date formats or obvious outliers?
The data-quality log below is the full answer — every issue found, how many rows, and the action.
We also look at numeric columns for implausible values.

In [27]:
quality = pd.DataFrame(quality_log)
display(quality[quality["rows_affected"] > 0])

display(df[["year", "month", "hour", "beat_num", "district_code", "ward_no", "community_code",
            "latitude", "longitude"]].describe().T[["min", "max", "mean"]])

,check,rows_affected,action
0,Line with unclosed quote (truncated export),2,"quote closed on file lines [498, 499]"
5,Date has no time of day (date only),498,time set to 00:00 – hour-of-day analysis is NO...
15,Missing location_desc,4,filled with 'UNKNOWN'
20,Duplicate ward_no in ward offices file,8,"kept the last (newest) record for wards [2, 6,..."
21,Duplicate beat_num in police beat file,2,identical rows – duplicates dropped


,min,max,mean
year,2012.0,2016.0,2015.965863
month,1.0,11.0,4.821285
hour,0.0,0.0,0.0
beat_num,111.0,2535.0,1115.927711
district_code,1.0,25.0,10.925703
ward_no,1.0,50.0,22.51004
community_code,1.0,77.0,37.11245
latitude,41.65325,42.019399,41.842657
longitude,-87.839659,-87.535758,-87.671066


**Time coverage.** A dataset can be perfectly formatted and still be unrepresentative. We check
how the crimes are spread over time.

In [28]:
by_day = df["date"].dt.date.value_counts()
top2_share = by_day.head(2).sum() / len(df) * 100
print(f"Date range: {df['date'].min():%d %b %Y} → {df['date'].max():%d %b %Y}")
print(f"Distinct dates: {by_day.size} | busiest 2 dates hold {top2_share:.1f}% of all crimes:")
display(by_day.head(5).rename("crimes").to_frame())
display(df["year"].value_counts().sort_index().rename("crimes").to_frame().T)
print(f"Rows with a real time of day: {(df['date'].dt.time != pd.Timestamp(0).time()).sum()}")
if top2_share > 50:
    print("\n⚠ The extract is concentrated on a few days: it looks like the records reported/updated in "
          "early May 2016 (date_of_update = May 2016 for every row). Older dates are crimes that were "
          "reported late, not a full history – keep this in mind for any year/month trend.")

Date range: 01 Jan 2012 → 04 May 2016
Distinct dates: 46 | busiest 2 dates hold 89.0% of all crimes:


,crimes
date,
2016-05-04,298
2016-05-03,145
2016-04-01,4
2015-01-01,2
2016-01-01,2


year,2012,2013,2014,2015,2016
crimes,2,1,1,4,490


Rows with a real time of day: 0

⚠ The extract is concentrated on a few days: it looks like the records reported/updated in early May 2016 (date_of_update = May 2016 for every row). Older dates are crimes that were reported late, not a full history – keep this in mind for any year/month trend.


## Save outputs for the next notebooks

In [29]:
df.to_pickle(cu.CLEAN_CRIME_FILE)
community.to_pickle(cu.OUTPUT_DIR / "community_lookup.pkl")
quality.to_csv(cu.OUTPUT_DIR / "data_quality_report.csv", index=False)
print(f"Saved {len(df):,} clean rows → {cu.CLEAN_CRIME_FILE.name}")
print(f"Raw rows: {len(raw):,}  →  clean rows: {len(df):,}  "
      f"({100 * (1 - len(df) / len(raw)):.2f}% removed)")

Saved 498 clean rows → chicago_crime_clean.pkl
Raw rows: 498  →  clean rows: 498  (0.00% removed)
